## Setting up model

In [2]:
from langchain.chat_models import ChatOpenAI

chat = ChatOpenAI(model_name="gpt-5-nano", temperature=1)


## Chatting with Message Objects

In [3]:
# from langchain.schema import HumanMessage, AIMessage, SystemMessage

# messages = [
#     SystemMessage(content="You are a geography expert and you only reply in Italian."),
#     AIMessage(content="Ciao, mi chiamo Palol!"),
#     HumanMessage(content="What is the distance between Mexico and Thailand. Also, what is your name?")
# ]

# chat.predict_messages(messages)

## Using ChatPromptTemplate

In [4]:
from langchain.prompts import PromptTemplate, ChatPromptTemplate

template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a geography expert and you only reply in {language}."),
        ("ai", "Ciao, mi chiamo {name}!"),
        ("human", "hat is the distance between {country_a} and {country_b}. Also, what is your name?")
    ]
)

prompt = template.format_messages(
    language="Greek",
    name="Socrates",
    country_a="Mexico",
    country_b="Thailand"
)

chat.predict_messages(prompt)

AIMessage(content='Απόσταση Μεξικό ↔ Ταϊλάνδη (ευθεία, μεγάλο κύκλο) για παράδειγμα Μεξικό Σίτι → Μπανγκόκ: περίπου 15,8 χιλιάδες χιλιόμετρα (περίπου 9,8 χιλιάδες μίλια). Η ακριβής απόσταση εξαρτάται από τα σημεία εκκίνησης/προσγείωσης που επιλέγετε.\n\nΤο όνομά μου: ChatGPT. Μπορείς να με αποκαλείς και Γεωγράφος.')

## OutputParser

In [12]:
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",")
        return list(map(str.strip, items))

In [13]:
template = ChatPromptTemplate.from_messages([
    ("system","You are a list generating machine. Everything you are asked will be answered with a comma separated list of max {max_items} in lowercase. Do NOT reply with anything else."),
    ("human", "{question}")
])

prompt = template.format_messages(
    max_items=10,
    question="What are the planets?"
)

result = chat.predict_messages(prompt)

p = CommaOutputParser()

p.parse(result.content)

['mercury', 'venus', 'earth', 'mars', 'jupiter', 'saturn', 'uranus', 'neptune']

## LCEL

In [16]:
chain = template | chat | CommaOutputParser()

chain.invoke({
    "max_items":5,
    "question":"What are the pokemons?"
})

['pikachu', 'bulbasaur', 'charizard', 'squirtle', 'eevee']

## Chaining Chains

In [25]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

chef_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a world-class international chef. You create easy to follow recipes for any type of cuisine with easy to find ingredients."),
    ("human", "I want to cook {cuisine} food.")
])

chef_chain = chef_prompt | chat

In [ ]:
veg_chef_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a vegetarian chef specialized on making traditional recipies vegetarian. You find alternative ingredients and explain their preparation. You don't radically modify the recipe. If there is no alternative for a food just say you don't know how to replace it."),
    ("human", "{recipe}")
])

veg_chain = veg_chef_prompt | chat

final_chain = {"recipe": chef_chain} | veg_chain

final_chain.invoke({"cuisine": "indian"})


Fantastic! Indian cooking is all about bold flavors and fresh ingredients. I can tailor to what you have, but here are three easy, beginner-friendly options (vegetarian and meat-friendly) plus a quick rice side to pair with everything.

Option 1: Chana Masala (Chickpea Curry) — quick, hearty, vegan
Ingredients (serves 2-3)
- 1 tbsp oil
- 1 medium onion, finely chopped
- 2–3 garlic cloves, minced
- 1 inch ginger, minced
- 1 green chili, optional, minced
- 1 tsp cumin seeds
- 1 cup canned crushed tomatoes (or 2 chopped tomatoes)
- 1 tsp ground coriander
- 1/2 tsp turmeric
- 1/2 tsp chili powder or paprika (adjust to taste)
- 1 tsp garam masala
- 1 can (15 oz) chickpeas, drained and rinsed
- 1/2 cup water (as needed)
- Salt to taste
- Fresh cilantro, chopped (for garnish)

Steps
1) Heat oil in a pan. Add cumin seeds and let them crackle 10–20 seconds.
2) Add onion; sauté until soft and translucent.
3) Stir in garlic, ginger, and chili; cook 1 minute.
4) Add coriander, turmeric, and chili 

AIMessageChunk(content='Love it. I can tailor these to what you have and your spice tolerance, without changing the spirit of the recipes.\n\nA few quick questions to tailor perfectly:\n- Are you strictly vegetarian, or ok with dairy (paneer/yogurt/ghee) or meat (poultry/beef) additions?\n- Spice level: mild, medium, or hot?\n- What do you actually have in your pantry? (Protein options like chickpeas, lentils, paneer; vegetables like onion, tomato, garlic, ginger, potatoes, cauliflower; staples like basmati rice, canned tomatoes, coconut milk; spices like cumin, coriander, turmeric, garam masala, chili powder.)\n- Any dietary restrictions (no onion/garlic, gluten-free, etc.)?\n\nIf you’d like, I can craft a single exact recipe right away based on common pantry items. For example, here’s a precise Chana Masala you can use as-is (serves 2–3):\n\nChana Masala (Chickpea Curry) — exact, serves 2–3\n- Oil: 15 ml (1 tablespoon)\n- Onion: 150 g, finely chopped\n- Garlic: 2–3 cloves, minced (ab

## FewShotPromptTemplate

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

examples = [
{
"question": "What do you know about France?",
"answer": """
Here is what I know:
Capital: Paris
Language: French
Food: Wine and Cheese
Currency: Euro
""",
},
{
"question": "What do you know about Italy?",
"answer": """
I know this:
Capital: Rome
Language: Italian
Food: Pizza and Pasta
Currency: Euro
""",
},
{
"question": "What do you know about Greece?",
"answer": """
I know this:
Capital: Athens
Language: Greek
Food: Souvlaki and Feta Cheese
Currency: Euro
""",
},
]

example_template = """
    Human: {question}
    AI: {answer}
"""

example_prompt = PromptTemplate.from_template(example_template)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"]
)

chain = prompt | chat

chain.invoke({
    "country": "Turkey"
})



I know this:
Capital: Ankara
Language: Turkish
Food: Kebap and Baklava
Currency: Turkish Lira

AIMessageChunk(content='I know this:\nCapital: Ankara\nLanguage: Turkish\nFood: Kebap and Baklava\nCurrency: Turkish Lira')

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

example_prompt = ChatPromptTemplate.from_messages( )

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"]
)

chain = prompt | chat

chain.invoke({
    "country": "Turkey"
})